# BJJ Pose Model Training — Colab

Upload `training_data.zip` and base model `.pt` to Google Drive under `roll_tracker_training/` before running.

In [ ]:
# Cell 1 — Setup
from google.colab import drive
drive.mount('/content/drive')

!pip install ultralytics -q

In [ ]:
# Cell 2 — Upload training data from Google Drive
DRIVE_PATH = "/content/drive/MyDrive/roll_tracker_training"
TRAINING_ZIP = f"{DRIVE_PATH}/training_data.zip"

!unzip -q {TRAINING_ZIP} -d /content/training_data_raw/

# The zip contains a 'combined/' directory — move contents up
import shutil
from pathlib import Path

unpacked = Path("/content/training_data_raw/combined")
target = Path("/content/training_data")
if unpacked.exists():
    if target.exists():
        shutil.rmtree(target)
    shutil.move(str(unpacked), str(target))
else:
    # Zip may have been created differently
    target = Path("/content/training_data_raw")
    target = target.rename("/content/training_data")

print(f"Training data at: {target}")
print(f"Images: {len(list(target.glob('images/*')))}")
print(f"Labels: {len(list(target.glob('labels/*')))}")

In [ ]:
# Cell 3 — Upload base model
import shutil
MODEL_NAME = "bjj-pose-r1.pt"  # Change per round
shutil.copy(f"{DRIVE_PATH}/{MODEL_NAME}", f"/content/{MODEL_NAME}")
print(f"Model ready: /content/{MODEL_NAME}")

In [ ]:
# Cell 4 — Fix dataset paths for Colab
import yaml
from pathlib import Path

dataset_yaml = Path("/content/training_data/dataset.yaml")
config = yaml.safe_load(dataset_yaml.read_text())

# Update path to Colab location
config["path"] = "/content/training_data"
dataset_yaml.write_text(yaml.dump(config, default_flow_style=False))

# Rewrite train.txt and val.txt with Colab paths
for split_file in ["train.txt", "val.txt"]:
    split_path = Path(f"/content/training_data/{split_file}")
    lines = split_path.read_text().strip().split("\n")
    new_lines = []
    for line in lines:
        filename = Path(line).name
        new_lines.append(f"/content/training_data/images/{filename}")
    split_path.write_text("\n".join(new_lines) + "\n")

print("Dataset paths updated for Colab")
print(f"Config: {yaml.dump(config)}")

In [ ]:
# Cell 5 — Train
from ultralytics import YOLO

# === CONFIGURE THESE PER ROUND ===
BASE_MODEL = "/content/bjj-pose-r1.pt"
FREEZE = 10
EPOCHS = 100
LR0 = 0.001
ROUND_NAME = "round2"
# =================================

model = YOLO(BASE_MODEL)

results = model.train(
    data="/content/training_data/dataset.yaml",
    epochs=EPOCHS,
    imgsz=640,
    batch=16,
    device=0,
    freeze=FREEZE,
    lr0=LR0,
    project=f"/content/training_runs/{ROUND_NAME}",
    name="train",
    exist_ok=True,
    save=True,
    plots=True,
    pose=12.0,
)

# Print final metrics
rd = getattr(results, "results_dict", {})
print(f"\n{'='*50}")
print(f"TRAINING COMPLETE \u2014 {ROUND_NAME}")
print(f"{'='*50}")
print(f"Box mAP50:     {rd.get('metrics/mAP50(B)', 0):.4f}")
print(f"Box mAP50-95:  {rd.get('metrics/mAP50-95(B)', 0):.4f}")
print(f"Pose mAP50:    {rd.get('metrics/mAP50(P)', 0):.4f}")
print(f"Pose mAP50-95: {rd.get('metrics/mAP50-95(P)', 0):.4f}")

In [ ]:
# Cell 6 — Save model to Drive
import shutil
from pathlib import Path

# Find best.pt
best_pt = Path(f"/content/training_runs/{ROUND_NAME}/train/weights/best.pt")
if not best_pt.exists():
    best_pt = list(Path(f"/content/training_runs/{ROUND_NAME}").rglob("best.pt"))[0]

# Copy to Drive
output_name = f"bjj-pose-{ROUND_NAME}.pt"
output_path = f"{DRIVE_PATH}/{output_name}"
shutil.copy2(best_pt, output_path)
print(f"Model saved to Drive: {output_path}")

# Also save training plots
plots_dir = best_pt.parent.parent
for plot_file in plots_dir.glob("*.png"):
    shutil.copy2(plot_file, f"{DRIVE_PATH}/{ROUND_NAME}_{plot_file.name}")
    print(f"Saved plot: {plot_file.name}")